# Layers

The core building blocks for graph neural networks in `kgcnn_torch` are located in `kgcnn_torch.layers`. These are standard PyTorch `nn.Module` classes (or pure functions) that implement the fundamental operations for GNNs:

1. **`gather`** -- Gathering node features along edges (pure functions).
2. **`aggr`** -- Aggregation layers for collecting edge messages to nodes.
3. **`conv`** -- Graph convolution implementations (GCN, SchNet, GIN, etc.).
4. **`pooling`** -- Graph-level readout pooling.
5. **`mlp`** -- Multi-layer perceptron with optional graph-aware normalization.
6. **`message`** -- Message passing base class.
7. **`geom`** -- Geometric operations (distances, angles, basis expansions).
8. **`activ`** -- Learnable activation layers.
9. **`norm`** -- Graph-aware normalization layers.
10. **`polynom`** -- Polynomial basis layers (Bessel, spherical harmonics).

All layers operate on the **disjoint graph representation** used by PyG, where a batch of graphs is stored as one large graph with batch indices.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

## Setup: Example Disjoint Graph

All `kgcnn_torch` layers work with the PyG disjoint graph format. Let us create a batch of 2 small graphs.

In [ ]:
# Graph 1: 3 nodes, 4 edges
# Graph 2: 2 nodes, 2 edges
# In disjoint form, node indices are offset:
#   Graph 1: nodes 0,1,2
#   Graph 2: nodes 3,4

node_features = torch.tensor([
    [1.0, 0.0],   # node 0 (graph 0)
    [0.0, 1.0],   # node 1 (graph 0)
    [1.0, 1.0],   # node 2 (graph 0)
    [2.0, 0.0],   # node 3 (graph 1)
    [0.0, 2.0],   # node 4 (graph 1)
])

# PyG convention: edge_index[0] = source, edge_index[1] = target
edge_index = torch.tensor([
    [0, 1, 1, 2,   3, 4],   # source
    [1, 0, 2, 1,   4, 3],   # target
])

batch = torch.tensor([0, 0, 0, 1, 1])  # Graph assignment per node
num_nodes = node_features.size(0)
batch_size = 2

print("Node features shape:", node_features.shape)
print("Edge index shape:", edge_index.shape)
print("Batch:", batch)

## Gather Operations

Gather functions select node features for each edge. They are pure functions (not `nn.Module`) for efficiency.

In [ ]:
from kgcnn_torch.layers.gather import (
    gather_nodes_outgoing,  # Source node features per edge
    gather_nodes_ingoing,   # Target node features per edge
    gather_nodes,           # Concatenated [target, source] per edge
    gather_state,           # Repeat graph-level state per node
)

# Gather source (outgoing) node features for each edge
x_source = gather_nodes_outgoing(node_features, edge_index)
print("Source features shape:", x_source.shape)  # (M, F)
print("Source features:\n", x_source)

# Gather target (ingoing) node features for each edge
x_target = gather_nodes_ingoing(node_features, edge_index)
print("\nTarget features shape:", x_target.shape)

# Concatenate [target, source] features
x_both = gather_nodes(node_features, edge_index)
print("\nConcatenated [target, source] shape:", x_both.shape)  # (M, 2*F)

In [ ]:
# Gather state: repeat graph-level features per node
graph_state = torch.tensor([[10.0, 20.0], [30.0, 40.0]])  # (B, F)
state_per_node = gather_state(graph_state, batch)
print("State per node:\n", state_per_node)  # (N, F)

## Scatter / Aggregation Operations

Scatter operations aggregate values by indices. These are the fundamental building blocks for message passing. The raw scatter functions are in `kgcnn_torch.ops.scatter`.

In [ ]:
from kgcnn_torch.ops.scatter import (
    scatter_reduce_sum,
    scatter_reduce_mean,
    scatter_reduce_max,
    scatter_reduce_softmax,
)

# Example: scatter sum of edge features to target nodes
edge_features = torch.tensor([
    [1.0], [2.0], [3.0], [4.0], [5.0], [6.0]
])
target_idx = edge_index[1]  # Target node indices

# Sum edge features at each target node
result_sum = scatter_reduce_sum(target_idx, edge_features, dim_size=num_nodes)
print("Scatter sum (per node):\n", result_sum)

# Mean aggregation
result_mean = scatter_reduce_mean(target_idx, edge_features, dim_size=num_nodes)
print("\nScatter mean (per node):\n", result_mean)

# Max aggregation
result_max = scatter_reduce_max(target_idx, edge_features, dim_size=num_nodes)
print("\nScatter max (per node):\n", result_max)

## Aggregation Layers

The `kgcnn_torch.layers.aggr` module provides `nn.Module` wrappers around scatter operations.

In [ ]:
from kgcnn_torch.layers.aggr import Aggregate, AggregateLocalEdges

# AggregateLocalEdges: aggregate edge features to target nodes
aggr = AggregateLocalEdges(pooling_method="sum")

# Create transformed edge features
edge_msg = nn.Linear(2, 8)(x_source)  # Transform source features
print("Edge messages shape:", edge_msg.shape)  # (M, 8)

# Aggregate to target nodes
agg_result = aggr(edge_msg, edge_index, num_nodes)
print("Aggregated per node shape:", agg_result.shape)  # (N, 8)

# Generic Aggregate layer
generic_aggr = Aggregate(pooling_method="mean")
result = generic_aggr(edge_msg, target_idx, dim_size=num_nodes)
print("Generic mean aggregate shape:", result.shape)

## Convolution Layers

The `kgcnn_torch.layers.conv` module provides graph convolution implementations.

In [ ]:
from kgcnn_torch.layers.conv import GCNConv, GINConv, SchNetInteraction

# GCN Convolution
gcn_conv = GCNConv(in_features=2, out_features=8, pooling_method="sum", activation="relu")
edge_weight = torch.ones(edge_index.size(1), 1)  # Uniform edge weights

x_updated = gcn_conv(node_features, edge_index, edge_weight)
print("GCN conv output shape:", x_updated.shape)  # (N, 8)

# GIN Convolution (sum aggregation + self-loop with epsilon)
gin_conv = GINConv(pooling_method="sum", epsilon_learnable=False)
x_gin = gin_conv(node_features, edge_index)
print("GIN conv output shape:", x_gin.shape)  # (N, 2) -- same as input

In [ ]:
# SchNet Interaction Block
# Requires node features of dimension 'units' and edge features of dimension 'edge_dim'
units = 16
edge_dim = 10

schnet_int = SchNetInteraction(
    units=units, edge_dim=edge_dim,
    activation="shifted_softplus",
    pooling_method="sum",
)

# Node features of size 'units', edge features of size 'edge_dim'
x_schnet = torch.randn(num_nodes, units)
ed_schnet = torch.randn(edge_index.size(1), edge_dim)  # e.g. Gaussian-expanded distances

x_out = schnet_int(x_schnet, ed_schnet, edge_index)
print("SchNet interaction output shape:", x_out.shape)  # (N, units)

## Pooling Layers

For graph-level embedding, node features are pooled per graph using the batch assignment.

In [ ]:
from kgcnn_torch.layers.pooling import PoolingNodes, PoolingWeightedNodes, PoolingEmbeddingAttention

# Sum pooling
pool = PoolingNodes(pooling_method="sum")
graph_embedding = pool(node_features, batch, batch_size)
print("Graph embedding (sum pooling):\n", graph_embedding)  # (B, F)

# Mean pooling
pool_mean = PoolingNodes(pooling_method="mean")
graph_embedding_mean = pool_mean(node_features, batch, batch_size)
print("\nGraph embedding (mean pooling):\n", graph_embedding_mean)

# Attention-based pooling
pool_attn = PoolingEmbeddingAttention(pooling_method="sum")
attention_logits = torch.randn(num_nodes, 1)
graph_attn = pool_attn(node_features, attention_logits, batch, batch_size)
print("\nGraph embedding (attention pooling):\n", graph_attn)

## MLP Layer

The `MLP` class supports optional normalization, dropout, and graph-aware normalization techniques.

In [ ]:
from kgcnn_torch.layers.mlp import MLP

# Basic MLP
mlp = MLP(
    units=[64, 32, 1],
    input_dim=2,
    activation=["relu", "relu", "linear"],
    use_bias=True,
)

out = mlp(node_features)
print("MLP output shape:", out.shape)  # (N, 1)

# MLP with dropout and batch normalization
mlp_norm = MLP(
    units=[32, 16],
    input_dim=2,
    activation="relu",
    use_dropout=True,
    dropout_rate=0.1,
    use_normalization=True,
    normalization_technique="layer",  # 'batch', 'layer', 'graph_batch', etc.
)

out_norm = mlp_norm(node_features)
print("MLP with normalization output shape:", out_norm.shape)

## Geometric Layers

The `kgcnn_torch.layers.geom` module provides operations for computing distances, angles, and basis function expansions.

In [ ]:
from kgcnn_torch.layers.geom import (
    compute_edge_distances,
    compute_edge_direction_normalized,
    GaussBasisLayer,
    BesselBasisLayer,
    CosCutOffEnvelope,
)

# Create 3D positions for our nodes
pos = torch.tensor([
    [0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [3.0, 0.0, 0.0],
    [3.0, 1.0, 0.0],
])

# Compute edge distances
distances = compute_edge_distances(pos, edge_index)
print("Edge distances shape:", distances.shape)  # (M, 1)
print("Distances:", distances.squeeze().tolist())

# Compute normalized direction vectors
directions = compute_edge_direction_normalized(pos, edge_index)
print("\nDirection vectors shape:", directions.shape)  # (M, 3)

In [ ]:
# Gaussian basis expansion (used by SchNet)
gauss = GaussBasisLayer(bins=20, distance=5.0, sigma=0.4)
gauss_features = gauss(distances)
print("Gaussian expansion shape:", gauss_features.shape)  # (M, 20)

# Bessel basis expansion (used by DimeNet, PAiNN)
bessel = BesselBasisLayer(num_radial=16, cutoff=5.0)
bessel_features = bessel(distances)
print("Bessel expansion shape:", bessel_features.shape)  # (M, 16)

# Cosine cutoff envelope
cutoff_env = CosCutOffEnvelope(cutoff=5.0)
envelope = cutoff_env(distances)
print("Cutoff envelope shape:", envelope.shape)  # (M, 1)
print("Envelope values:", envelope.squeeze().tolist())

## Message Passing Base Class

The `MessagePassingBase` class provides a template for custom message passing layers. Subclasses implement `message_function` and `update_nodes`.

In [ ]:
from kgcnn_torch.layers.message import MessagePassingBase


class MyMessageNN(MessagePassingBase):
    """Custom message passing layer."""

    def __init__(self, in_dim, out_dim, **kwargs):
        super().__init__(**kwargs)
        self.dense = nn.Linear(2 * in_dim, out_dim)
        self.act = nn.ReLU()

    def message_function(self, n_in, n_out, edges=None):
        # n_in: target node features, n_out: source node features
        return self.act(self.dense(torch.cat([n_in, n_out], dim=-1)))

    def update_nodes(self, nodes, aggregated):
        return nodes + aggregated  # Residual update


# Use the custom layer (input dim must match node_features dim)
msg_layer = MyMessageNN(in_dim=2, out_dim=2)
x_updated = msg_layer(node_features, edge_index)
print("Custom message passing output shape:", x_updated.shape)

## Building a Custom GNN from Primitives

Here is a complete example of building a custom GNN using the primitive layers.

In [ ]:
class CustomGNN(nn.Module):
    """A custom GNN built from kgcnn_torch layer primitives.
    
    Architecture:
      1. Embed atomic numbers
      2. Compute distances and expand with Gaussian basis
      3. 3x message passing with edge features
      4. Sum pooling -> output MLP
    """

    def __init__(self, units=64, depth=3, gauss_bins=20, cutoff=5.0):
        super().__init__()
        self.node_emb = nn.Embedding(95, units)
        self.gauss = GaussBasisLayer(bins=gauss_bins, distance=cutoff)

        self.edge_mlps = nn.ModuleList()
        self.node_mlps = nn.ModuleList()
        self.aggrs = nn.ModuleList()

        for _ in range(depth):
            self.edge_mlps.append(nn.Sequential(
                nn.Linear(2 * units + gauss_bins, units),
                nn.SiLU(),
            ))
            self.aggrs.append(AggregateLocalEdges(pooling_method="sum"))
            self.node_mlps.append(nn.Sequential(
                nn.Linear(units + units, units),
                nn.SiLU(),
            ))

        self.pooling = PoolingNodes(pooling_method="sum")
        self.output_mlp = MLP(units=[units, 1], input_dim=units, activation=["silu", "linear"])

    def forward(self, data):
        z, pos, edge_index, batch_idx = data.z, data.pos, data.edge_index, data.batch
        num_nodes = z.size(0)
        batch_size = int(batch_idx.max().item()) + 1

        # Node embedding
        x = self.node_emb(z.long())

        # Edge features from distances
        dist = compute_edge_distances(pos, edge_index)
        ed = self.gauss(dist)  # (M, gauss_bins)

        # Message passing
        for edge_mlp, aggr, node_mlp in zip(self.edge_mlps, self.aggrs, self.node_mlps):
            x_src = gather_nodes_outgoing(x, edge_index)
            x_tgt = x[edge_index[1]]
            msg = edge_mlp(torch.cat([x_tgt, x_src, ed], dim=-1))
            agg = aggr(msg, edge_index, num_nodes)
            x = node_mlp(torch.cat([x, agg], dim=-1))

        # Graph-level readout
        out = self.pooling(x, batch_idx, batch_size)
        return self.output_mlp(out)


# Test the custom GNN
from torch_geometric.data import Data, Batch

test_data = Batch.from_data_list([
    Data(z=torch.tensor([6, 6, 8]), pos=torch.randn(3, 3),
         edge_index=torch.tensor([[0,1,1,2],[1,0,2,1]])),
    Data(z=torch.tensor([7, 6]), pos=torch.randn(2, 3),
         edge_index=torch.tensor([[0,1],[1,0]])),
])

gnn = CustomGNN(units=32, depth=2)
with torch.no_grad():
    out = gnn(test_data)
print("Custom GNN output:", out)

> **NOTE**: You can find this page as a Jupyter notebook in the `docs/source` directory of the kgcnn-torch repository.